# Incident Response Runbook: Reconnaissance - Phishing for Information

**Tactic:** Reconnaissance
**Technique:** Phishing for Information (T1598)
**Severity:** MEDIUM

## Overview

This runbook provides step-by-step incident response procedures for Phishing for Information activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add the project root to the Python path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()

# Query Splunk for phishing reconnaissance indicators
print(f"\n[QUERY] Searching Splunk for phishing reconnaissance indicators...")
splunk_query = '''
index=* (sourcetype=email OR sourcetype=mail) (phishing OR spearphishing OR "phish" OR "social engineering")
OR (sourcetype=web OR sourcetype=access_combined) (phishing OR "phish" OR "credential harvest" OR "fake login")
OR (sourcetype=WinEventLog:Security EventCode=4625) "phishing" OR "spear" OR "social"
OR (sourcetype=network) "phishing" OR "phish" OR "credential" OR "harvest"
| eval sender_ip=coalesce(src_ip, clientip, sender_ip)
| eval target_user=coalesce(recipient, user, target_user, username)
| eval phishing_type=case(match(_raw, "spear"), "spear_phishing", match(_raw, "whaling"), "whaling", match(_raw, "vishing"), "vishing", true(), "phishing")
| stats count by sender_ip, target_user, phishing_type, subject, _time
| where count > 3
'''

try:
    splunk_results = splunk.search_events(splunk_query, timeframe="-24h")
    print(f"   Found {len(splunk_results)} potential phishing reconnaissance events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    splunk_results = []

# Extract affected systems and phishing activities
affected_systems = []
phishing_activities = []
splunk_indicators = []

for event in splunk_results:
    system_info = {
        'sender_ip': event.get('sender_ip', 'unknown'),
        'target_user': event.get('target_user', 'unknown'),
        'phishing_type': event.get('phishing_type', 'phishing'),
        'subject': event.get('subject', ''),
        'attempt_count': event.get('count', 0),
        'last_seen': event.get('_time', detection_time)
    }
    affected_systems.append(system_info)

    # Extract indicators
    if event.get('sender_ip') and event['sender_ip'] != 'unknown':
        splunk_indicators.append({
            'type': 'ip',
            'value': event.get('sender_ip'),
            'context': f"Phishing reconnaissance from {event.get('sender_ip')} targeting {event.get('target_user', 'users')}"
        })

# Query CrowdStrike for phishing detections
print(f"\n[QUERY] Checking CrowdStrike for phishing detections...")
try:
    cs_detections = crowdstrike.get_detections(
        filter="technique:'Phishing for Information'",
        start_time="-24h"
    )
    cs_phishing_sources = []
    for detection in cs_detections:
        phishing_info = {
            'sender_ip': detection.get('sender_ip', ''),
            'target_user': detection.get('target_user', ''),
            'detection_id': detection.get('detection_id', ''),
            'technique': detection.get('technique', ''),
            'severity': detection.get('severity', 0),
            'phishing_details': detection.get('phishing_details', {})
        }
        cs_phishing_sources.append(phishing_info)

        # Merge with Splunk data
        existing_system = next((s for s in affected_systems if s['sender_ip'] == phishing_info['sender_ip']), None)
        if existing_system:
            existing_system.update(phishing_info)
        else:
            affected_systems.append(phishing_info)

    print(f"   Found {len(cs_phishing_sources)} CrowdStrike phishing detections")
except Exception as e:
    print(f"   CrowdStrike query failed: {e}")
    cs_phishing_sources = []

# Enrich with threat intelligence from MISP
print(f"\n[ENRICHMENT] Checking MISP for related threat intelligence...")
misp_results = []
try:
    for indicator in splunk_indicators:
        misp_search = misp.search_iocs(indicator['value'])
        if misp_search:
            misp_results.extend(misp_search)
            print(f"   Found threat intelligence for {indicator['value']}")
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    incident_data = {
        'title': f'Phishing Reconnaissance Incident - {len(affected_systems)} phishing attempts',
        'description': f'Detected phishing reconnaissance activities targeting {len(affected_systems)} users',
        'severity': 'MEDIUM',
        'tactic': 'Reconnaissance',
        'technique': 'Phishing for Information (T1598)',
        'affected_systems': affected_systems,
        'indicators': splunk_indicators,
        'threat_intel': misp_results,
        'detection_time': detection_time
    }
    incident_id = iris.create_case(incident_data)
    print(f"   Created IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"TEMP-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n Detection complete:")
print(f"  - {len(affected_systems)} phishing reconnaissance attempts identified")
print(f"  - {len(splunk_indicators)} phishing indicators found")
print(f"  - {len(misp_results)} threat intelligence matches")
print(f"  - IRIS case {incident_id} created")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
isolated_hosts = []
blocked_ips = []
disabled_accounts = []

# Isolate affected systems
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            isolation_result = crowdstrike.isolate_host(system['device_id'])
            if isolation_result.get('status') == 'isolated':
                isolated_hosts.append(system.get('hostname', system.get('sender_ip', 'unknown')))
                print(f"   Isolated host: {system.get('hostname', system.get('sender_ip', 'unknown'))}")
    except Exception as e:
        print(f"   Host isolation failed for {system.get('hostname', system.get('sender_ip', 'unknown'))}: {e}")

# Block phishing source IPs
print(f"\n[ACTION] Blocking phishing source IPs...")
unique_ips = set()
for system in affected_systems:
    if system.get('sender_ip') and system['sender_ip'] != 'unknown':
        unique_ips.add(system['sender_ip'])

for ip in unique_ips:
    try:
        block_result = shuffle.block_ip_address(ip)
        if block_result.get('status') == 'blocked':
            blocked_ips.append(ip)
            print(f"   Blocked phishing IP: {ip}")
    except Exception as e:
        print(f"   IP blocking failed for {ip}: {e}")

# Disable compromised user accounts
print(f"\n[ACTION] Disabling compromised user accounts...")
unique_users = set()
for system in affected_systems:
    if system.get('target_user') and system['target_user'] != 'unknown':
        unique_users.add(system['target_user'])

for user in unique_users:
    try:
        disable_result = shuffle.disable_user_account(user)
        if disable_result.get('status') == 'disabled':
            disabled_accounts.append(user)
            print(f"   Disabled account: {user}")
    except Exception as e:
        print(f"   Account disable failed for {user}: {e}")

# Enable enhanced monitoring
print(f"\n[ACTION] Enabling enhanced monitoring...")
monitoring_rules = []
try:
    # Enable CrowdStrike phishing monitoring
    cs_monitoring = crowdstrike.enable_enhanced_monitoring('phishing_recon')
    if cs_monitoring.get('status') == 'enabled':
        monitoring_rules.append('CrowdStrike phishing monitoring')
        print(f"   Enabled CrowdStrike phishing reconnaissance monitoring")

    # Enable Splunk phishing correlation rules
    splunk_monitoring = splunk.enable_correlation_rule('phishing_reconnaissance')
    if splunk_monitoring.get('status') == 'enabled':
        monitoring_rules.append('Splunk phishing correlation')
        print(f"   Enabled Splunk phishing reconnaissance correlation rules")
except Exception as e:
    print(f"   Enhanced monitoring setup failed: {e}")

# Update IRIS case with containment actions
print(f"\n[ACTION] Updating IRIS case with containment actions...")
try:
    containment_summary = {
        'isolated_hosts': len(isolated_hosts),
        'blocked_ips': len(blocked_ips),
        'disabled_accounts': len(disabled_accounts),
        'monitoring_enabled': len(monitoring_rules),
        'containment_time': containment_time
    }
    iris.update_case(incident_id, 'containment_complete', containment_summary)
    print(f"   Updated IRIS case {incident_id} with containment results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_hosts)} hosts isolated")
print(f"  - {len(blocked_ips)} phishing IPs blocked")
print(f"  - {len(disabled_accounts)} accounts disabled")
print(f"  - {len(monitoring_rules)} enhanced monitoring rules enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

eradication_time = datetime.now().isoformat()
terminated_processes = []
quarantined_files = []
removed_artifacts = []

# Terminate malicious processes
print(f"\n[ACTION] Terminating malicious processes...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            processes = crowdstrike.get_suspicious_processes(system['device_id'], 'phishing_recon')
            for proc in processes:
                if proc.get('process_name') in ['phishing.exe', 'recon_tool.exe', 'malware_scanner.exe']:
                    term_result = crowdstrike.terminate_process(system['device_id'], proc['process_id'])
                    if term_result.get('status') == 'terminated':
                        terminated_processes.append(f"{system.get('hostname', 'unknown')}:{proc['process_name']}")
                        print(f"   Terminated process: {proc['process_name']} on {system.get('hostname', 'unknown')}")
    except Exception as e:
        print(f"   Process termination failed for {system.get('hostname', 'unknown')}: {e}")

# Quarantine phishing-related files
print(f"\n[ACTION] Quarantining phishing-related files...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            files = crowdstrike.get_phishing_files(system['device_id'])
            for file_path in files:
                quarantine_result = crowdstrike.quarantine_file(system['device_id'], file_path)
                if quarantine_result.get('status') == 'quarantined':
                    quarantined_files.append(f"{system.get('hostname', 'unknown')}:{file_path}")
                    print(f"   Quarantined file: {file_path} on {system.get('hostname', 'unknown')}")
    except Exception as e:
        print(f"   File quarantine failed for {system.get('hostname', 'unknown')}: {e}")

# Remove phishing artifacts from network
print(f"\n[ACTION] Removing phishing artifacts from network...")
try:
    # Remove phishing domains from DNS
    phishing_domains = misp.get_phishing_domains()
    for domain in phishing_domains:
        try:
            remove_result = shuffle.remove_dns_record(domain)
            if remove_result.get('status') == 'removed':
                removed_artifacts.append(f"DNS record: {domain}")
                print(f"   Removed DNS record for phishing domain: {domain}")
        except Exception as e:
            print(f"   DNS removal failed for {domain}: {e}")

    # Clean up phishing email traces
    email_cleanup = splunk.clean_phishing_emails()
    if email_cleanup.get('status') == 'cleaned':
        removed_artifacts.append(f"Email traces: {email_cleanup.get('count', 0)} messages")
        print(f"   Cleaned up {email_cleanup.get('count', 0)} phishing email traces")
except Exception as e:
    print(f"   Network artifact removal failed: {e}")

# Update threat intelligence
print(f"\n[ACTION] Updating threat intelligence...")
try:
    eradication_iocs = {
        'phishing_ips': [system.get('sender_ip') for system in affected_systems if system.get('sender_ip') != 'unknown'],
        'phishing_domains': phishing_domains,
        'compromised_accounts': [system.get('target_user') for system in affected_systems if system.get('target_user') != 'unknown']
    }
    misp.share_iocs(eradication_iocs, 'phishing_reconnaissance_eradicated')
    print(f"   Shared eradication IOCs with MISP threat intelligence")
except Exception as e:
    print(f"   Threat intelligence update failed: {e}")

# Update IRIS case with eradication actions
print(f"\n[ACTION] Updating IRIS case with eradication actions...")
try:
    eradication_summary = {
        'terminated_processes': len(terminated_processes),
        'quarantined_files': len(quarantined_files),
        'removed_artifacts': len(removed_artifacts),
        'eradication_time': eradication_time
    }
    iris.update_case(incident_id, 'eradication_complete', eradication_summary)
    print(f"   Updated IRIS case {incident_id} with eradication results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(terminated_processes)} malicious processes terminated")
print(f"  - {len(quarantined_files)} phishing files quarantined")
print(f"  - {len(removed_artifacts)} network artifacts removed")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

recovery_time = datetime.now().isoformat()
restored_systems = []
recreated_accounts = []
restored_services = []

# Restore isolated systems
print(f"\n[ACTION] Restoring isolated systems...")
for system in affected_systems:
    try:
        if 'device_id' in system and system.get('hostname') in isolated_hosts:
            restore_result = crowdstrike.restore_host(system['device_id'])
            if restore_result.get('status') == 'restored':
                restored_systems.append(system.get('hostname', 'unknown'))
                print(f"   Restored system: {system.get('hostname', 'unknown')}")
    except Exception as e:
        print(f"   System restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Recreate disabled user accounts
print(f"\n[ACTION] Recreating disabled user accounts...")
for user in disabled_accounts:
    try:
        recreate_result = shuffle.recreate_user_account(user)
        if recreate_result.get('status') == 'recreated':
            recreated_accounts.append(user)
            print(f"   Recreated account: {user}")
    except Exception as e:
        print(f"   Account recreation failed for {user}: {e}")

# Restore affected services
print(f"\n[ACTION] Restoring affected services...")
try:
    # Restore email services
    email_restore = shuffle.restore_email_service()
    if email_restore.get('status') == 'restored':
        restored_services.append('Email service')
        print(f"   Restored email service")

    # Restore web proxy services
    proxy_restore = shuffle.restore_web_proxy()
    if proxy_restore.get('status') == 'restored':
        restored_services.append('Web proxy')
        print(f"   Restored web proxy service")

    # Restore DNS services
    dns_restore = shuffle.restore_dns_service()
    if dns_restore.get('status') == 'restored':
        restored_services.append('DNS service')
        print(f"   Restored DNS service")
except Exception as e:
    print(f"   Service restoration failed: {e}")

# Validate system integrity
print(f"\n[ACTION] Validating system integrity...")
integrity_checks = []
try:
    for system in affected_systems:
        if 'device_id' in system:
            integrity_result = crowdstrike.validate_system_integrity(system['device_id'])
            if integrity_result.get('status') == 'valid':
                integrity_checks.append(system.get('hostname', 'unknown'))
                print(f"   System integrity validated: {system.get('hostname', 'unknown')}")
            else:
                print(f"   System integrity issues detected: {system.get('hostname', 'unknown')}")
except Exception as e:
    print(f"   Integrity validation failed: {e}")

# Re-enable monitoring
print(f"\n[ACTION] Re-enabling standard monitoring...")
try:
    # Restore normal CrowdStrike monitoring
    cs_restore = crowdstrike.restore_standard_monitoring()
    if cs_restore.get('status') == 'restored':
        print(f"   Restored standard CrowdStrike monitoring")

    # Restore normal Splunk correlation rules
    splunk_restore = splunk.restore_standard_correlation()
    if splunk_restore.get('status') == 'restored':
        print(f"   Restored standard Splunk correlation rules")
except Exception as e:
    print(f"   Monitoring restoration failed: {e}")

# Update IRIS case with recovery actions
print(f"\n[ACTION] Updating IRIS case with recovery actions...")
try:
    recovery_summary = {
        'restored_systems': len(restored_systems),
        'recreated_accounts': len(recreated_accounts),
        'restored_services': len(restored_services),
        'integrity_validated': len(integrity_checks),
        'recovery_time': recovery_time
    }
    iris.update_case(incident_id, 'recovery_complete', recovery_summary)
    print(f"   Updated IRIS case {incident_id} with recovery results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems restored")
print(f"  - {len(recreated_accounts)} accounts recreated")
print(f"  - {len(restored_services)} services restored")
print(f"  - {len(integrity_checks)} systems validated")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident
print("\n" + "=" * 60)
print("STEP 5: Post-Incident")
print("=" * 60)

post_incident_time = datetime.now().isoformat()

# Generate incident report
print(f"\n[ACTION] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'technique': 'Phishing Reconnaissance (T1598)',
        'detection_time': detection_time,
        'containment_time': containment_time,
        'eradication_time': eradication_time,
        'recovery_time': recovery_time,
        'post_incident_time': post_incident_time,
        'affected_systems': len(affected_systems),
        'isolated_hosts': len(isolated_hosts),
        'blocked_ips': len(blocked_ips),
        'disabled_accounts': len(disabled_accounts),
        'terminated_processes': len(terminated_processes),
        'quarantined_files': len(quarantined_files),
        'removed_artifacts': len(removed_artifacts),
        'restored_systems': len(restored_systems),
        'recreated_accounts': len(recreated_accounts),
        'restored_services': len(restored_services),
        'integrity_validated': len(integrity_checks),
        'threat_intelligence_shared': True,
        'lessons_learned': [
            'Enhanced phishing email filtering needed',
            'User training on phishing recognition required',
            'Improved monitoring for reconnaissance activities',
            'Regular threat intelligence updates implemented'
        ]
    }

    # Save report to IRIS
    iris.save_incident_report(incident_id, incident_report)
    print(f"   Incident report saved to IRIS case {incident_id}")

    # Share anonymized report with MISP
    misp.share_incident_report(incident_report, 'phishing_reconnaissance_response')
    print(f"   Anonymized incident report shared with MISP community")
except Exception as e:
    print(f"   Report generation failed: {e}")

# Conduct lessons learned session
print(f"\n[ACTION] Conducting lessons learned analysis...")
try:
    lessons_learned = {
        'what_went_well': [
            'Rapid detection through Splunk correlation rules',
            'Effective containment via CrowdStrike isolation',
            'Comprehensive threat intelligence enrichment',
            'Automated response workflows functioned correctly'
        ],
        'what_could_improve': [
            'Earlier detection of reconnaissance phase',
            'More granular user training on phishing indicators',
            'Enhanced email gateway filtering',
            'Better integration between security tools'
        ],
        'preventive_measures': [
            'Implement advanced email authentication (DMARC/SPF/DKIM)',
            'Deploy endpoint detection for reconnaissance tools',
            'Regular security awareness training',
            'Enhanced network segmentation',
            'Automated threat hunting for phishing campaigns'
        ]
    }

    iris.add_lessons_learned(incident_id, lessons_learned)
    print(f"   Lessons learned documented in IRIS case {incident_id}")
except Exception as e:
    print(f"   Lessons learned analysis failed: {e}")

# Update security controls
print(f"\n[ACTION] Updating security controls...")
try:
    # Update Splunk correlation rules
    splunk.update_phishing_rules(incident_report)
    print(f"   Updated Splunk phishing detection rules")

    # Update CrowdStrike policies
    crowdstrike.update_phishing_policies(incident_report)
    print(f"   Updated CrowdStrike phishing prevention policies")

    # Update Shuffle workflows
    shuffle.update_phishing_workflows(incident_report)
    print(f"   Updated Shuffle phishing response workflows")
except Exception as e:
    print(f"   Security controls update failed: {e}")

# Close incident
print(f"\n[ACTION] Closing incident...")
try:
    closure_summary = {
        'closure_time': post_incident_time,
        'incident_status': 'resolved',
        'follow_up_actions': [
            'Monitor for similar phishing campaigns',
            'Conduct user awareness training',
            'Review and update email security policies',
            'Schedule threat hunting exercises'
        ],
        'metrics': {
            'time_to_detect': 'Calculated from detection_time',
            'time_to_contain': 'Calculated from containment_time',
            'time_to_eradicate': 'Calculated from eradication_time',
            'time_to_recover': 'Calculated from recovery_time',
            'affected_assets': len(affected_systems)
        }
    }

    iris.close_case(incident_id, closure_summary)
    print(f"   IRIS case {incident_id} closed successfully")
except Exception as e:
    print(f"   Incident closure failed: {e}")

print(f"\n Post-incident activities complete:")
print(f"  - Incident report generated and shared")
print(f"  - Lessons learned documented")
print(f"  - Security controls updated")
print(f"  - Incident case closed")

print(f"\n Phishing reconnaissance incident response completed successfully!")
print(f"   Incident ID: {incident_id}")
print(f"   Total duration: {len(affected_systems)} systems protected")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
